# CE541E08 — Unit 2 · Day 11 — if-else: Pressure Check, Froude Classification and Self-cleansing
| | |
|---|---|
| **Course** | CE541E08 |
| **Department** | Civil Engineering · Christ University |
| **Instructor** | Dr. Arpan Pradhan |
| **Unit** | Unit 2 |
| **Session** | Day 11 of 45 |
| **CO** | CO2 |
| **Topics** | if-else · pressure check · Froude subcritical/supercritical · self-cleansing velocity · hydraulic jump |
---
> Read the explanation before each code block. Check the expected output. Run the cell and verify. Then try the small challenge.
---

In [ ]:
student_name = "Your Full Name"
roll_number  = "2024XXXXXX"
github_repo  = "https://github.com/your-username/CE541E08-2026"
session      = "Day 11"
print(f"CE541E08 | {student_name} | {roll_number} | {session}")

---
## Section 1 — if-else in Engineering Design

The if-else pattern is the most common decision structure in engineering code. Every design check is a decision: does this value meet the criterion? The if block runs when it does, the else block runs when it doesn't. Exactly one branch always executes.

---
## Code Block 1 — Pipe Pressure Check

### What this code does

We check whether the operating pressure in a PN10 water main exceeds its rated maximum, printing either a safety confirmation or a warning with the excess pressure.

### Why each step is taken

**`if operating_pressure_kPa <= max_allowable_kPa:`:**
The check is "less than or equal to" — the pipe is safe if pressure is at OR below its rating. The `else` block runs when pressure exceeds the rating.

**`excess = operating_pressure_kPa - max_allowable_kPa`:**
Computing excess pressure inside the else block — only makes sense when pressure is too high, so this calculation is conditional.

### Algorithm

```
1. operating=720 kPa, max=700 kPa
2. if 720 <= 700:  → False → else block
3. excess = 720 - 700 = 20 kPa
4. Print warning with excess
```

### Expected output

```
Pressure 720 kPa: EXCEEDS safe limit ✗
  Excess: 20 kPa above PN10 rating (700 kPa)
  Action: Review pipe class — upgrade to PN16.
```

In [ ]:
operating_pressure_kPa = 720    # try: 200, 500, 720, 900
max_allowable_kPa      = 700   # PN10 pipe rating

if operating_pressure_kPa <= max_allowable_kPa:
    print(f"Pressure {operating_pressure_kPa} kPa: WITHIN safe limits ✓")
    margin = max_allowable_kPa - operating_pressure_kPa
    print(f"  Safety margin: {margin} kPa below rated maximum")
else:
    print(f"Pressure {operating_pressure_kPa} kPa: EXCEEDS safe limit ✗")
    excess = operating_pressure_kPa - max_allowable_kPa
    print(f"  Excess: {excess} kPa above PN10 rating ({max_allowable_kPa} kPa)")
    print("  Action: Review pipe class — upgrade to PN16.")

### 🔁 Try this

Change to `operating_pressure_kPa = 700` (exactly at the limit). Does it pass or fail?

The condition is `<=`, so exactly at the limit should pass. Verify this.

---
## Code Block 2 — Froude with Hydraulic Jump Sequent Depth

### What this code does

We classify flow as subcritical or supercritical, and for supercritical flow we compute the sequent depth for the hydraulic jump — the depth downstream of a stilling basin.

### Why each step is taken

**`if Fr < 1.0:` / `else:`:**
The classification determines whether a hydraulic jump can occur. A jump is only possible from supercritical to subcritical — so sequent depth is only relevant when Fr > 1.

**`y2 = (y/2) * (math.sqrt(1 + 8*Fr**2) - 1)`:**
The Belanger equation for sequent depth. `8*Fr**2` uses `**` for squaring. The parentheses around the sqrt argument are essential — without them, only `1` would be inside the square root.

**Why compute y2 only in the else block:**
y2 only has physical meaning when Fr > 1. Computing it for subcritical flow would give a nonsensical value.

### Algorithm

```
1. V=2.8, y=0.6, g=9.81
2. Fr = 2.8/sqrt(9.81×0.6) = 1.153
3. 1.153 < 1.0 → False → else block
4. y2 = (0.6/2)*(sqrt(1+8×1.153²)−1) = 1.060 m
```

### Expected output

```
Velocity V    : 2.8 m/s
Flow depth y  : 0.6 m
Froude number : 1.153

SUPERCRITICAL flow (Fr > 1.0)
  Shooting flow — provides energy for hydraulic jump.
  Sequent depth y2 = 1.060 m
  Basin minimum length ≈ 6.360 m
```

In [ ]:
import math

V = 2.8    # m/s  try: 0.5, 1.5, 2.8, 4.5
y = 0.6    # m
g = 9.81

Fr = V / math.sqrt(g * y)

print(f"Velocity V    : {V} m/s")
print(f"Flow depth y  : {y} m")
print(f"Froude number : {Fr:.3f}")
print()

if Fr < 1.0:
    print("SUBCRITICAL flow (Fr < 1.0)")
    print("  Tranquil, slow flow — easy to control.")
    print("  Suitable for: irrigation canals, river channels")
else:
    print("SUPERCRITICAL flow (Fr > 1.0)")
    print("  Shooting flow — provides energy for hydraulic jump.")
    y2 = (y/2) * (math.sqrt(1 + 8*Fr**2) - 1)
    print(f"  Sequent depth y2 = {y2:.3f} m")
    print(f"  Basin minimum length ≈ {6*y2:.3f} m")

### 🔁 Try this

For what velocity V does Fr = 1.0 (critical flow) when y = 0.6 m?

Solve V_c = sqrt(g×y) = sqrt(9.81×0.6) and compute it in Python. Verify with the Froude formula.

---
## Code Block 3 — Self-cleansing Velocity

### What this code does

We check whether a sewer pipe velocity meets the self-cleansing criterion (minimum 0.6 m/s) and quantify the deficit if it fails.

### Why each step is taken

**`if V >= 0.6:`:** Pipes must carry flow fast enough to prevent sediment settlement. Below 0.6 m/s, particles settle and the pipe gradually blocks.

**`deficit = 0.6 - V`:** Computed only when V < 0.6. This gives the engineer a quantified shortfall — they need to increase slope or reduce diameter to achieve this additional velocity.

### Expected output (V=0.45 m/s)

```
V = 0.45 m/s: Below self-cleansing velocity ✗
  Deficit: 0.15 m/s below minimum (0.6 m/s)
  Risk: sediment deposition → blockage over time.
  Recommendation: increase slope or reduce diameter.
```

In [ ]:
V = 0.45   # m/s  try: 0.3, 0.45, 1.8, 3.5

if V >= 0.6:
    print(f"V = {V} m/s: Above self-cleansing velocity ✓")
    print("  Sediment will not accumulate in this pipe.")
else:
    print(f"V = {V} m/s: Below self-cleansing velocity ✗")
    deficit = 0.6 - V
    print(f"  Deficit: {deficit:.2f} m/s below minimum (0.6 m/s)")
    print("  Risk: sediment deposition → blockage over time.")
    print("  Recommendation: increase slope or reduce diameter.")

### 🔁 Try this

For V = 0.45 m/s and D = 0.3 m, what slope S would bring velocity to exactly 0.6 m/s using Manning's?

Rearrange Manning's: S = (V×n / R^(2/3))²  where n=0.013, R=D/4=0.075.

Compute S and express as 1:X (slope ratio).

---
## Session Summary — if-else Engineering Applications

| Application | Condition | if block | else block |
|---|---|---|---|
| Pressure check | `p <= p_max` | Safety margin | Excess + action |
| Froude class | `Fr < 1.0` | Subcritical info | Supercritical + y2 |
| Self-cleansing | `V >= 0.6` | Pass | Deficit + advice |
| Velocity range | `0.6<=V<=3.0` | Acceptable | REVIEW |

---
## Day 11 Assignment

Channel: B=6.0m, y=0.5m, V=3.2m/s, g=9.81. Compute Fr. If supercritical, compute y2.

### ▶ Assignment cell

In [ ]:
import math
B=6.0; y=0.5; V=3.2; n=0.018; S=0.008; g=9.81

Fr = ???

if ???:
    print(f"Fr={Fr:.3f}: SUBCRITICAL — no energy dissipator required")
else:
    y2 = ???   # sequent depth
    print(f"Fr={Fr:.3f}: SUPERCRITICAL")
    print(f"Sequent depth y2 = {y2:.3f} m")
    print("Provide stilling basin — minimum length ≈ 6 × y2")

---
- [ ] Run all cells — verify outputs match expected outputs
- [ ] Complete the assignment cell
- [ ] Upload: `Unit2_LoopsDecisions/CE541E08_U2_Day11.ipynb`
- [ ] Commit: `Day 11 assignment completed`

*CE541E08 · Civil Engineering · Christ University · 2026-27 · Dr. Arpan Pradhan*